# 第3章 AIエージェント 最小構成

LLM自身に「次に呼ぶツール(関数)」を選ばせる **エージェント** の最小例を体験します。

**所要時間**: 約20分

> 💡 **モデルに関する補足**
> 本ハンズオンでは Amazon Nova Lite を使用します。Nova シリーズは `ChatBedrockConverse` 経由で Tool Calling に対応していますが、`langchain-aws` のバージョンや Nova モデル自体の挙動の違いにより、ツール呼び出しがうまく動かないケースが報告されたこともあります。
> もしこの章で `agent.invoke(...)` がエラーになったり、ツールが呼ばれず最終回答が不自然な場合は、`.env` の `BEDROCK_CHAT_MODEL_ID` を以下に変更して試してください:
>
> ```
> BEDROCK_CHAT_MODEL_ID=anthropic.claude-3-haiku-20240307-v1:0
> ```
>
> ⚠️ Anthropic 製モデルは自動有効化されますが、**初回利用時に一度だけ「使用フォーム」(use case詳細)の提出が必要** です(数分で承認されます)。詳細は [docs/00_setup.md](../docs/00_setup.md#anthropic製モデルを使う場合オプション) を参照。

## 0. 準備

In [ ]:
import os
from dotenv import load_dotenv

# .env を読み込む(これまでと同じ)
load_dotenv()

# 第3章では LLM のみ使用(埋め込みは不要)
AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

## 1. ツール(=ただのPython関数)を定義

`@tool` デコレータを付けると、LangChainがLLMに渡せるツールスキーマに変換してくれます。

**`@tool` の働き**
1. 関数のシグネチャ(引数名・型・デフォルト値)を読み取り、JSONスキーマを自動生成
2. 関数の **docstring を「ツールの説明」** として LLM に渡す
3. ツール名は関数名がそのまま使われる

LLM はこの「名前・説明・引数スキーマ」を見て、どのツールを呼ぶべきかを判断します。
docstring が雑だと LLM が誤ったツールを選びがちなので、**docstring は丁寧に書く** のがコツです。

**型ヒントが重要な理由**
- `timezone: str = "Asia/Tokyo"` のような型ヒントは、JSONスキーマの `type: "string"` 等に変換される
- LLM は「文字列を渡せばいいんだな」と理解し、正しい型で呼び出してくれる

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
# @tool は普通の関数を「LLMに渡せるツール」に変換するデコレータ
from langchain_core.tools import tool


# @tool デコレータでツール化。関数本体は普通の Python 関数のまま動く
@tool
def get_current_time(timezone: str = "Asia/Tokyo") -> str:
    """指定したタイムゾーンの現在時刻を ISO 8601 文字列で返す。
    timezone は IANA 形式(例: 'Asia/Tokyo', 'UTC')。"""
    # ↑ この docstring が LLM への "ツール説明" として渡される
    return datetime.now(ZoneInfo(timezone)).isoformat(timespec="seconds")


@tool
def add_numbers(a: float, b: float) -> float:
    """2つの数値を加算した結果を返す。"""
    return a + b


# LLM に渡すために、ツールをリストにまとめる
tools = [get_current_time, add_numbers]

# 各ツールに自動付与された name と description を確認
# (name = 関数名、 description = docstring)
for t in tools:
    print(f"- {t.name}: {t.description}")

## 2. エージェントを構築

- `create_agent`: langchain 1.x 標準のエージェントファクトリ。モデル・ツール・システムプロンプトを渡すだけで Agent が完成する。
- 内部は **LangGraph のグラフ**(LLM呼び出し → ツール実行 → LLM呼び出し → ... のループ)として自動構築される。

**旧 API との対応**

langchain 0.x の `create_tool_calling_agent`(Agent本体)+ `AgentExecutor`(実行ループ)の2分割は、langchain 1.x では `create_agent` 1つに統合されました。「次に何を呼ぶか決める」判断と「ツールを実行してループを回す」実行が、ひとまとめになっています。

**入出力の形式**

入力も出力も `{"messages": [...]}` 形式(OpenAI 互換のメッセージ列)です。戻り値の `messages` には [人間の質問 → AIのツール呼び出し → ツール結果 → ... → 最終回答] が時系列で全部入ります。これは第5章の LangGraph とまったく同じ形式なので、次章にそのまま繋がります。

> 補足: 旧版で必要だった `MessagesPlaceholder("agent_scratchpad")`(ツール呼び出し履歴の差し込み口)は、`create_agent` では内部で自動処理されるため、自分で書く必要はありません。

In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent  # langchain 1.x 標準のエージェントファクトリ

# LLM 準備(これまでと同じ)
llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# システムプロンプト(エージェントの振る舞いの指示)。
# 旧版では ChatPromptTemplate + MessagesPlaceholder("agent_scratchpad") を自分で組んだが、
# create_agent では scratchpad(ツール呼び出し履歴の差し込み)は内部で自動処理される。
system_prompt = (
    "あなたはツールを使いこなすアシスタントです。\n"
    "必要に応じてツールを呼び、その結果を踏まえて日本語で簡潔に答えてください。"
)

# create_agent: モデル・ツール・システムプロンプトを渡すだけでエージェントが完成する。
# 内部では LangGraph のグラフ(LLM → ツール → LLM のループ)が自動で組まれる。
agent = create_agent(llm, tools=tools, system_prompt=system_prompt)

## 3. 実行: 時刻を尋ねる

LLMが `get_current_time` を呼ぶはずです。戻り値の `messages` を順に見ると、ツール選択 → 実行 → 結果を踏まえた回答、という流れが追えます。

**メッセージ履歴の読み方**

`result["messages"]` には次のようなメッセージが時系列で入ります:

```
Human:  いま東京は何時ですか?                       ← ユーザーの質問
Ai:     (tool_calls) get_current_time(timezone=...) ← LLMが選んだツールと引数
Tool:   2026-05-27T...                              ← 関数の戻り値
Ai:     [最終回答]                                   ← LLMが結果をまとめて返した答え
```

`m.pretty_print()` は各メッセージを役割(Human / Ai / Tool)つきで見やすく表示するヘルパーです。最終回答だけ欲しいときは `result["messages"][-1].content` を見ます。

In [ ]:
# 入力は {"messages": [...]} 形式。各メッセージは {"role": ..., "content": ...} の dict。
# role は "user"(ユーザー) / "assistant"(AI) / "system"(指示) など。
result = agent.invoke({"messages": [{"role": "user", "content": "いま東京は何時ですか?"}]})

# 戻り値も {"messages": [...]} 形式。messages には
# [人間の質問, AIのツール呼び出し, ツール結果, 最終回答] が時系列で全部入る。
print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()

print("\n=== 最終回答 ===")
print(result["messages"][-1].content)  # [-1] が最終回答

## 4. 実行: 計算を頼む

今度は `add_numbers` が選ばれるはずです。

In [ ]:
# 計算が必要な質問 → add_numbers が選ばれるはず
# (LLMは数値計算が苦手なので、Toolに任せた方が正確という典型的な例)
result = agent.invoke({"messages": [{"role": "user", "content": "123.4 と 56.78 を足すといくつ?"}]})
print("=== 最終回答 ===")
print(result["messages"][-1].content)

## 5. 実行: ツールが不要な質問

ツールを呼ばずに LLM が直接答えるパターンも観察します。

In [ ]:
# 一般知識で答えられる質問 → LLMはツールを呼ばず、自分の知識で直接回答する
# messages にツール呼び出し(Ai の tool_calls)や Tool メッセージが現れないことを確認
result = agent.invoke({"messages": [{"role": "user", "content": "フランスの首都はどこ?"}]})
print("=== 全メッセージ履歴 ===")
for m in result["messages"]:
    m.pretty_print()
print("\n=== 最終回答 ===")
print(result["messages"][-1].content)

## まとめ

- `@tool` で Python 関数をエージェントが使えるツールに変換できる
- `create_agent`(langchain 1.x 標準)で **LLM ↔ Tool のループ** が回る。内部は LangGraph のグラフ
- 入出力は `{"messages": [...]}` 形式。履歴をたどれば、どのツールが呼ばれたか観察できる
- LLMは質問に応じて、ツールを呼ぶか / 自分で答えるかを判断する

次は [04 まとめ](../04_wrapup/) で、ここから先の学習リンクを紹介します。